<div style="background-color:#0B3C5D; padding:25px; border-radius:10px;">

![](https://raw.githubusercontent.com/wateraccounting/WaPORMOOC/main/images/banner_notebooks_WaPOR4Global.png)


<div style="text-align:center; margin-top:15px;">
<h3 style="color:#D9E6F2; margin-bottom:5px;">
<i>Module Two</i> – <b>Data Access</b>
</h3>

<p style="color:white; margin-top:5px;">
<b>Notebook:</b> Download WaPOR data to local machine &nbsp; | &nbsp;
<b>Estimated time:</b> 2 hours &nbsp; | &nbsp;
<b>Instructor:</b> Dr. Solomon Seyoum
</p>
</div>

</div>


<div style="border-left:6px solid #1D70B8; background-color:#0B3C5D; padding:15px; border-radius:6px">

### 🎯 Learning Objectives

When you complete this notebook, you will be able to:

- Explain the different ways how WaPOR data can be accessed  
- Understand the code in the notebook
- Access WaPOR data and download the data needed for module 3  
- Modify the code to access and download the data for your need    

</div>

<div style="background-color:#0B3C5D; padding:15px; border-radius:6px">

### 📝 Prerequisites

✔ [Introduction to WaPORv3](https://ocw.un-ihe.org/course/view.php?id=263)  
✔ Geospatial data concepts  
✔ [Python basics for geeospatial data analysis](https://ocw.un-ihe.org/course/view.php?id=272)

</div>

<div style="background-color:#0B3C5D; padding:18px; border-radius:6px">

## 📖 WaPOR data access

### WaPOR data can be accessed in several ways, in this notebook we will focus on using the WaPOR v3 API

The scripts used in this notebook are built with the WaPOR Version 3 API to download and preprocess different types for data for a selected area(s) of interest and store them in different formats (raster or timeseries):

*   Download raster images for the area of interest and store in a netCDF format
*   Download timeseries from polygons and store in a csv format
*   Download timeseries for points and store in a csv format
*   Download timeseries for a polygon and masked for land cover or crop type and store as csv file

NOTE: The scripts also explain how to download AgERA5 data (available from 1979) useful for long term climatic analyses


</div>


---

###  🔀 Workflow Overview
🗺

| Step | Task | Purpose |
|------|------|---------|
| 1 | Environment Setup | importing required packages
| 2 | Input Data Preparation | Set up the input required to run the download script |
| 3 | Running the download script | Downlaad the data |
| 4 | Check| Check if the required data is downloaded based on the specification |
| 5 | Optional - Visualize | Visualize the downloaded data |

---

## ⚙️ 1. Environment Setup
<p style="margin-top:1px;">
	Some Python packages are required that may not be included in a basic Python installation. Run the cells below to set up your environment.
	</p>

In [1]:
# --- Install the required packages
%%capture
!pip install netCDF4
!pip install geopandas
!pip install shapely
!pip install rasterio
!pip install xarray
!pip install dask
!pip install rioxarray
!pip install "dask[distributed]"

In [2]:
# import packages
import os
import shutil
import zipfile
from glob import glob
import xarray as xr
import pandas as pd

In this Notebook we will be using a package called **download_WaPORv3_data** to access the WaPOR API and download WaPORv3 data.

[Download](https://github.com/wateraccounting/WaPORMOOC/blob/main/package/download_WaPOR_v3_data.py) the python code from github, and upload to this session. Ensure that the directory path where the file is located is updated.

NOTE: It is also possible to download AgERA5 data using this package as will be explained in the exercises.

In [4]:
#--- Define directory path where download_WaPOR_v3_data.py is stored
dir_code = '/content'

# --- import required packages
import time
import sys
import importlib
sys.path.append(dir_code) #add folder with local modules to system paths
import download_WaPOR_v3_data as wdl
wld = importlib.reload(wdl)


## ⚙️ 2. Input data preparation
<p style="margin-top:1px;">
It is also good practice to use an organized folder structure, with your input data in one folder and your output data in another. The data folder can contain different formats of input and output files. <br><br>
	In the next cells, we will create the necessary folders.

In [11]:
dir_input_data = '/content/WaPOR4GC/input_data'
if not os.path.exists(dir_input_data):
    os.makedirs(dir_input_data)
dir_input_data

dir_output_data = '/content/WaPOR4GC/output_data'
if not os.path.exists(dir_output_data):
    os.makedirs(dir_output_data)
dir_input_data, dir_output_data

('/content/WaPOR4GC/input_data', '/content/WaPOR4GC/output_data')

For the upcoming exercises you will be using different data sets to download the data, needed for the exercises in Module 3:

*   Iraq_admin1 (containing the location of the 18 governorates of Iraq)
*   Locations of wells Shamamouk
*   Erbil_gov
*   ESA_LCC_ map (land cover map obtained from ESA for the Erbil governorate and resampled to 20m resolution)
*   shapefile of farm fields

Download this data from the [data folder](https://github.com/wateraccounting/WaPOR4Global/tree/main/data) in the WaPOR4Global repository on github. Upload this data into the input_data folder.



### Information needed to run the download script
The download script has a similar structure as what you are familiar with (as explained in the Introduction WaPORv3 MOOC). It requires the following information from the user:
<ul style="margin-top:0px;">
  <li>The folder path where the data should be saved</li>
  <li>The area of interest as a shapefile </li>
  <li>A list of the WaPOR data products needed, including level, variable name, and frequency</li>
  <li>The start and end dates for the requested data</li>
  <li>The desired data format: There are four possibilities
   <ul>
      <li>raster data in NetCDF format</li>
      <li>point data for one or multiple locations</li>
      <li>Zonal statistics using polygons</li>
      <li>Zonal statistics by raster: for example statistics per land cover class</li>
    </ul>
  </li>
   We will use example to download data for each format below.   
</ul>

### 3a - Download raster data AOI
This notebook, similar to the one in the Introduction to WaPORv3 MOOC downloads raster data from WaPOR. The main difference is that it compiles the data into a netCDF file. If you want individual TIFF files please use the [WaPOR_download script](https://github.com/wateraccounting/WaPORMOOC/blob/main/1_WaPOR_download_colab/Download_WaPORv3_Data.ipynb) from the previous MOOC.

Steps:
*   Upload a shapefile or geojson to the input data folder
*   Create a folder by putting the name in the first line
*   Update the relative path in the second line
*   Select the products and period of the data you want to download
*   Run the cell
*   Check the folder for the downloaded file

NOTE: data_type is set as **'raster'**
The example is for the Wad Helal area in Gezira irrigation scheme.

The geojson file of the Wad Helal area can be found in the [data folder in the WaPOR4Global repository](https://github.com/wateraccounting/WaPOR4global/tree/main/data) (*Wad_Helal.geojson*).





In [19]:
project_foldr = f"{dir_output_data}/Gezira"
region = r"/content/WaPOR4GC/data/Wad_Helal.geojson"
products = [ "L3-AETI-D", "L3-NPP-D"]
period = ["2022-10-01", "2023-04-30"]
data_type = "raster"

#  --- Run the download script
# %cd /content
tt = time.time()
wdl.wapor_dl(region, products, period, project_foldr, data_type=data_type)

elapsed = time.time() - tt
print(
	">> Time elapsed up to downloading the required data : "
	+ "{0:.1f}".format(elapsed)
	+ " s"
)


INFO:distributed.scheduler:State start
INFO:distributed.scheduler:  Scheduler at: inproc://172.28.0.12/156/41
INFO:distributed.scheduler:  dashboard at:  http://172.28.0.12:42675/status
INFO:distributed.scheduler:Registering Worker plugin shuffle
INFO:distributed.worker:      Start worker at: inproc://172.28.0.12/156/44
INFO:distributed.worker:         Listening to:          inproc172.28.0.12
INFO:distributed.worker:          Worker name:                          0
INFO:distributed.worker:         dashboard at:          172.28.0.12:46685
INFO:distributed.worker:Waiting to connect to: inproc://172.28.0.12/156/41
INFO:distributed.worker:-------------------------------------------------
INFO:distributed.worker:              Threads:                          2
INFO:distributed.worker:               Memory:                   9.31 GiB
INFO:distributed.worker:      Local Directory: /tmp/dask-scratch-space/worker-vjlius6y
INFO:distributed.worker:------------------------------------------------

No running Dask scheduler found, starting a new LocalCluster...
Started new LocalCluster with 1 workers.
Processing raster
using L3-AETI-A as template.
the AOI is within Gezira, Sudan
processing L3-AETI-D ...
processing L3-NPP-D ...


INFO:distributed.scheduler:Remove client Client-89c6da40-2466-11f1-809c-0242ac1c000c
INFO:distributed.core:Received 'close-stream' from inproc://172.28.0.12/156/46; closing.
INFO:distributed.scheduler:Remove client Client-89c6da40-2466-11f1-809c-0242ac1c000c
INFO:distributed.scheduler:Close client connection: Client-89c6da40-2466-11f1-809c-0242ac1c000c


>> Time elapsed up to downloading the required data : 81.0 s


If the above cell runs successfully, then you will see the netCDF files created in the folder "output_data/Gezira/nc" one for each variable.

### 3b - Download timeseries of point locations

This notebook allows for downloading timeseries for point locations. Upload a shapefile with point locations and select the product and period to download. Notice that the data_type is changed to **'point'**.

Note: To label the columns in the file where the data will be saved, assign the name of the column to points_col_name. Otherwise the index will be used.

We will download daily data for precipitation and reference ET for a number of stations accross Africa. The geojson file with the met station locations is located in the data folder of the WaPOR4Global repository (*met_stations.geojson*).

In [23]:

project_foldr = f"{dir_output_data}/Africa"
region = r"/content/WaPOR4GC/data/met_stations.geojson"
products = ["L1-PCP-E","L1-RET-E"]
period = ["2023-01-01", "2023-12-31"]
data_type = "point"

tt = time.time()
points_col_name = "Station_code"  ## The name of the points id
wdl.wapor_dl(
				region,
				products,
				period,
				project_foldr,
				data_type=data_type,
				points_col_name=points_col_name,
		)

elapsed = time.time() - tt
print(
	">> Time elapsed up to downloading the required data : "
	+ "{0:.1f}".format(elapsed)
	+ " s"
)

Reusing existing Dask client.
Sampling points using 'Station_code'
processing L1-PCP-E ...
14 locations are within the bounds of L1-PCP-E
processing L1-RET-E ...
14 locations are within the bounds of L1-RET-E


INFO:distributed.scheduler:Remove client Client-1f23ab92-2468-11f1-809c-0242ac1c000c
INFO:distributed.core:Received 'close-stream' from inproc://172.28.0.12/156/54; closing.
INFO:distributed.scheduler:Remove client Client-1f23ab92-2468-11f1-809c-0242ac1c000c
INFO:distributed.scheduler:Close client connection: Client-1f23ab92-2468-11f1-809c-0242ac1c000c


>> Time elapsed up to downloading the required data : 142.6 s


If the above cell runs successfully, then you will see CSV files created in the folder "output_data/Shammamuk/csvs" one for each variable.

### 3b - Download timeseries for polygons using zonal statistics.

To download timeseries for polygons, the following notebook can be used. Note that the data type is now set at zonal_stat, which requires that the type of statistic needs to be provided, the default is **'Mean'**.

If the shapefile of the AIO has a name column to identify the polygons, it will  be used to lable the columns in the file where the data will be saved. Otherwise the index will be used.

In this Notebook, we will download the WaPOR AETI and NPP data for the farmer fields in the Wad Helal area (case study of the MOOC on python for geospatial analyses).


---


In [18]:
project_foldr = f"{dir_data}/Gezira"
region = r"/content/WaPOR4GC/data/WH_Fields.geojson"
products = ["L3-AETI-D","L3-NPP-D"]
# products = ["AgERA5-PCP-M"] # ["AgERA5-RET-M"]
period = ["2022-10-01", "2023-04-30"]
data_type = "zonal_stat"

tt = time.time()
polygons_col_name = "admin1_name"
stat = "mean"  # possible stats include max, min, median
wdl.wapor_dl(
    region,
    products,
    period,
    project_foldr,
    data_type=data_type,
    stat=stat,
    polygons_col_name=polygons_col_name,
)

elapsed = time.time() - tt
print(
	">> Time elapsed up to downloading the required data : "
	+ "{0:.1f}".format(elapsed)
	+ " s"
)

INFO:distributed.scheduler:State start
INFO:distributed.scheduler:  Scheduler at: inproc://172.28.0.12/156/33
INFO:distributed.scheduler:  dashboard at:  http://172.28.0.12:43585/status
INFO:distributed.scheduler:Registering Worker plugin shuffle
INFO:distributed.worker:      Start worker at: inproc://172.28.0.12/156/36
INFO:distributed.worker:         Listening to:          inproc172.28.0.12
INFO:distributed.worker:          Worker name:                          0
INFO:distributed.worker:         dashboard at:          172.28.0.12:44947
INFO:distributed.worker:Waiting to connect to: inproc://172.28.0.12/156/33
INFO:distributed.worker:-------------------------------------------------
INFO:distributed.worker:              Threads:                          2
INFO:distributed.worker:               Memory:                   9.31 GiB
INFO:distributed.worker:      Local Directory: /tmp/dask-scratch-space/worker-snpgjt5v
INFO:distributed.worker:------------------------------------------------

No running Dask scheduler found, starting a new LocalCluster...
Started new LocalCluster with 1 workers.
Zonal stats on 'admin1_name' using stat 'mean'
Reprojecting shapefile from EPSG:32636 to EPSG:4326...
Shapefile reprojected successfully.
the AOI is within Gezira, Sudan
processing L3-AETI-D ...
187 polygons are within the bounds of L3-AETI-D
processing L3-NPP-D ...
187 polygons are within the bounds of L3-NPP-D


INFO:distributed.scheduler:Remove client Client-ee11dad9-2463-11f1-809c-0242ac1c000c
INFO:distributed.core:Received 'close-stream' from inproc://172.28.0.12/156/38; closing.
INFO:distributed.scheduler:Remove client Client-ee11dad9-2463-11f1-809c-0242ac1c000c
INFO:distributed.scheduler:Close client connection: Client-ee11dad9-2463-11f1-809c-0242ac1c000c


>> Time elapsed up to downloading the required data : 37.2 s


If the above cell runs successfully, then you will see CSV files created in the folder "data/Gezira/csvs" one for each variable. See how easy this was?

## **Exercise**:
It is also possible to download climatic data (PCP and RET) from AgERA5 using the same script. The advantage is that this data is available since 1979.

Change the line products by replacing it with the reference to AgERA5 data as illustrated below

```
products = ["AgERA5-PCP-M", "AgERA5-RET-M"]
```

---

For module 3, you have to download **monthly** AgERA5 Precipitation and Reference ET data for the 18 different governorates in Iraq for the period 1980-2025. The geojson file of the Iraqi governorates can be found in the [data folder in the WaPOR4Global repository](https://github.com/wateraccounting/WaPOR4global/tree/main/data) (*irq_admin1.geojson*). Adapt the script above in such a way that it download the required data. Dont forget to save the data on your computer before you close colab.

### 3d - Download data - Zonal statistics by raster: for example, statistics per land cover class.

N.B. The path to the raster to define the zones should be provided. In this example we use the land cover raster for the Erbil area in Iraq.

The raster file which defines the zones (for example the land cover raster) may have a different resolution than the data to be downloaded. In this case we can chose either to resample the data to be downloaded to match the LCC raster or resample the LCC to match the resolution of the data to be downloaded.

In [27]:
project_foldr = f"{dir_output_data}/Erbil1"
region = r"/content/WaPOR4GC/data/Shammamuk.geojson"
products = [ "L2-AETI-D"]
period = ["2023-06-01", "2023-12-31"]
data_type = "sample_by_lcc"

tt = time.time()
stat = "mean"  # possible stats include max,
lcc_raster_path = r"/content/WaPOR4GC/data/ESA_LC_2021_Erbil_20m.tif"
wdl.wapor_dl(
    region,
    products,
    period,
    project_foldr,
    data_type=data_type,
    stat=stat,
    lcc_raster_path=lcc_raster_path,
    use_lcc_resolution="no",  # if yes, the data will be resampled to mach the lcc raster
)

elapsed = time.time() - tt
print(
	">> Time elapsed up to downloading the required data : "
	+ "{0:.1f}".format(elapsed)
	+ " s"
)

INFO:distributed.scheduler:State start
INFO:distributed.scheduler:  Scheduler at: inproc://172.28.0.12/156/73
INFO:distributed.scheduler:  dashboard at:  http://172.28.0.12:37171/status
INFO:distributed.scheduler:Registering Worker plugin shuffle
INFO:distributed.worker:      Start worker at: inproc://172.28.0.12/156/76
INFO:distributed.worker:         Listening to:          inproc172.28.0.12
INFO:distributed.worker:          Worker name:                          0
INFO:distributed.worker:         dashboard at:          172.28.0.12:44683
INFO:distributed.worker:Waiting to connect to: inproc://172.28.0.12/156/73
INFO:distributed.worker:-------------------------------------------------
INFO:distributed.worker:              Threads:                          2
INFO:distributed.worker:               Memory:                   9.31 GiB
INFO:distributed.worker:      Local Directory: /tmp/dask-scratch-space/worker-j_zztolu
INFO:distributed.worker:------------------------------------------------

No running Dask scheduler found, starting a new LocalCluster...
Started new LocalCluster with 1 workers.
Bounding box: (43.35883333333334, 35.43225, 45.078583333333334, 37.32033333333334)


INFO:distributed.scheduler:Remove client Client-cd0de080-2469-11f1-809c-0242ac1c000c
INFO:distributed.core:Received 'close-stream' from inproc://172.28.0.12/156/78; closing.
INFO:distributed.scheduler:Remove client Client-cd0de080-2469-11f1-809c-0242ac1c000c
INFO:distributed.scheduler:Close client connection: Client-cd0de080-2469-11f1-809c-0242ac1c000c


>> Time elapsed up to downloading the required data : 11.4 s


If the above cell runs successfully, then you will see CSV files created in the folder "data/Erbil/csvs" one for each variable.

### Download the data folder to local machine



In [ ]:
source_folder = dir_output_data
destination_folder = "/content"
zip_name = f"Downloaded.zip"

# Run zip command
!zip -r "{destination_folder}/{zip_name}" "{source_folder}"

from google.colab import files
files.download(f"{destination_folder}/{zip_name}")

# 📊 Exercise

Please prepare shapefiles of your area of interest to download raster and point data. Upload the shapefiles  and download raster data and point data for your area of interest.

